In [7]:
!pip install py3Dmol rdkit-pypi
!pip install "numpy<2"
!pip install deepchem
!pip install rdkit-pypi
!pip install py3Dmol

ERROR: Could not find a version that satisfies the requirement py3Dmol (from versions: none)
ERROR: No matching distribution found for py3Dmol

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\msi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\msi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\msi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\msi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement py3Dmol (from versions: none)
ERROR: No matching distribution found for py3Dmol

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\msi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ------------------ Imports ------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
import io, base64
from PIL import Image
from IPython.display import HTML, display
import warnings

import deepchem as dc
from deepchem.feat import CircularFingerprint
from deepchem.molnet import load_delaney
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

from rdkit import rdBase
rdBase.DisableLog('rdApp.warning')
warnings.filterwarnings('ignore', category=UserWarning, module='deepchem')

# ------------------ Function for table ------------------
def create_molecule_table(dataset, n=12, label_type='regression', task_name=None):
    data = []
    num_to_display = min(n, len(dataset))

    for i in range(num_to_display):
        smi = dataset.ids[i]
        mol = Chem.MolFromSmiles(smi)

        img_html = "Invalid SMILES"
        if mol is not None:
            try:
                img = Draw.MolToImage(mol, size=(200, 200))
                buf = io.BytesIO()
                img.save(buf, format="PNG")
                img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
                img_html = f'<img src="data:image/png;base64,{img_b64}" width="200" height="200"/>'
            except Exception as e:
                img_html = f"Image Generation Error: {e}"

        label_val = dataset.y[i]
        label_str = "N/A"
        if label_val is not None and hasattr(label_val, '__len__') and len(label_val) > 0:
            val_to_format = label_val[0]
            if label_type == 'regression':
                try:
                    label_str = f"{val_to_format:.3f}"
                except (TypeError, ValueError):
                    label_str = str(val_to_format)
            elif label_type == 'classification':
                try:
                    label_str = "Active" if int(val_to_format) == 1 else "Inactive"
                except (TypeError, ValueError):
                    label_str = str(val_to_format)
            else:
                label_str = str(val_to_format)
        elif label_val is not None:
            label_str = str(label_val)

        property_label = f"{task_name}: {label_str}" if task_name else label_str
        data.append([smi, img_html, property_label])

    df = pd.DataFrame(data, columns=["SMILES", "Structure", "Property"])
    return df
# ------------------ Load ESOL dataset ------------------
tasks_esol, datasets_esol_raw, transformers_esol = load_delaney(
    featurizer='Raw',
    splitter='random',
    reload=True
)
train_esol_raw, valid_esol_raw, test_esol_raw = datasets_esol_raw

print(f"ESOL Dataset Tasks: {tasks_esol}")
print(f"Number of training samples: {len(train_esol_raw)}")
print(f"Number of validation samples: {len(valid_esol_raw)}")
print(f"Number of test samples: {len(test_esol_raw)}")

y_all_esol = np.concatenate([train_esol_raw.y, valid_esol_raw.y, test_esol_raw.y])
print("\nSolubility (logS) Statistics:")
print(f"Mean: {np.mean(y_all_esol):.2f}")
print(f"Min: {np.min(y_all_esol):.2f}")
print(f"Max: {np.max(y_all_esol):.2f}")
print(f"Std Dev: {np.std(y_all_esol):.2f}")

plt.figure(figsize=(10, 6))
plt.hist(y_all_esol, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(np.mean(y_all_esol), color='red', linestyle='dashed', linewidth=1, label=f'Mean: {np.mean(y_all_esol):.2f}')
plt.xlabel('logS (mol/L)')
plt.ylabel('Frequency')
plt.title('Distribution of Aqueous Solubility in ESOL Dataset')
plt.legend()
plt.show()

df_esol = create_molecule_table(train_esol_raw, n=10, label_type='regression', task_name='logS')
try:
    display(HTML(df_esol.to_html(escape=False, index=False)))
except NameError:
    print(df_esol.to_string())

# ------------------ ECFP featurization & modeling ------------------
featurizer_esol = CircularFingerprint(radius=2, size=1024)
tasks_esol, datasets_esol, transformers_esol = load_delaney(
    featurizer=featurizer_esol,
    splitter='random',
    transformers=[],
    reload=True
)
train_esol, valid_esol, test_esol = datasets_esol

rf_esol = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_esol = dc.models.SklearnModel(rf_esol)
model_esol.fit(train_esol)

preds_esol = model_esol.predict(test_esol).flatten()
y_true_esol = test_esol.y.flatten()

mse_esol = mean_squared_error(y_true_esol, preds_esol)
rmse_esol = np.sqrt(mse_esol)
print(f"Test MSE: {mse_esol:.4f}")
print(f"Test RMSE: {rmse_esol:.4f}")

plt.figure(figsize=(8, 8))
plt.scatter(y_true_esol, preds_esol, alpha=0.7)
plt.xlabel("True Solubility (logS)")
plt.ylabel("Predicted Solubility (logS)")
plt.title("Random Forest Prediction on ESOL Dataset")
lims = [min(np.min(y_true_esol), np.min(preds_esol)), max(np.max(y_true_esol), np.max(preds_esol))]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
plt.xlim(lims)
plt.ylim(lims)
plt.grid(True)
plt.text(lims[0]*0.9+lims[1]*0.1, lims[1]*0.9+lims[0]*0.1, f'RMSE = {rmse_esol:.3f}', ha='left', va='top')
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [4]:
# ------------------ 3D visualization for all molecules ------------------
import py3Dmol
from rdkit.Chem import AllChem

def show_3d_structure(smiles, title="Molecule"):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    AllChem.UFFOptimizeMolecule(mol)
    mb = Chem.MolToMolBlock(mol)

    viewer = py3Dmol.view(width=400, height=300)
    viewer.addModel(mb, "mol")
    viewer.setStyle({'stick': {}})
    viewer.setBackgroundColor('0xeeeeee')
    viewer.zoomTo()
    viewer.setTitle(title)
    viewer.show()

print("\n💧 Displaying molecules in a table with 3D viewers:")

# Loop through first 10 molecules
for i, smi in enumerate(train_esol_raw.ids[:10]):
    # Get logS value
    label_val = train_esol_raw.y[i]
    if label_val is not None and hasattr(label_val, '__len__') and len(label_val) > 0:
        prop_str = f"{label_val[0]:.3f}"
    else:
        prop_str = "N/A"

    mol_title = f"Train molecule #{i+1}"

    # Create a mini DataFrame for each row and display it
    df_row = pd.DataFrame([[smi, prop_str]], columns=["SMILES", "logS"])
    display(HTML(df_row.to_html(escape=False, index=False)))

    # Display the 3D viewer right after
    show_3d_structure(smi, title=mol_title)


ModuleNotFoundError: No module named 'py3Dmol'